### 02 - Pretraitement
#### HumanForYou - Attrition ML

Objectif : fusionner les 3 fichiers RH, supprimer les colonnes non informatives, imputer les valeurs manquantes et encoder `Attrition`.

- **Entrees** : `general_data.csv`, `employee_survey_data.csv`, `manager_survey_data.csv`
- **Sortie** : `data/processed/attrition_merged_base.csv`


#### 1. Import et chargement

Cette cellule charge les trois jeux de donn*es RH qui serviront de base au pr*traitement.


In [1]:
# Imports pour le pretraitement tabulaire et les controles rapides.
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

general = pd.read_csv(os.path.join(RAW_DIR, 'general_data.csv'))
employee_survey = pd.read_csv(os.path.join(RAW_DIR, 'employee_survey_data.csv'))
manager_survey = pd.read_csv(os.path.join(RAW_DIR, 'manager_survey_data.csv'))

print(f'general         : {general.shape}')
print(f'employee_survey : {employee_survey.shape}')
print(f'manager_survey  : {manager_survey.shape}')


general         : (4410, 24)
employee_survey : (4410, 4)
manager_survey  : (4410, 3)


#### 2. Merge sur EmployeeID

Cette *tape fusionne les tables autour de `EmployeeID` afin d*obtenir un dataset unique par employ*.


In [2]:
# Fusion des trois tables sur `EmployeeID` pour obtenir une base unique.
df = general.merge(employee_survey, on='EmployeeID', how='inner') \
            .merge(manager_survey, on='EmployeeID', how='inner')

print(f'Shape apres merge : {df.shape}')
assert df.shape[0] == 4410, f'Nombre de lignes inattendu : {df.shape[0]}'


Shape apres merge : (4410, 29)


#### 3. Suppression des colonnes non informatives

`EmployeeCount`, `Over18` et `StandardHours` ont une seule valeur unique - aucun pouvoir discriminant.


In [3]:
# Suppression de colonnes constantes ou non informatives.
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours']
for col in cols_to_drop:
    print(f'{col} : {df[col].nunique()} valeur(s) unique(s) -> {df[col].unique()}')

df = df.drop(columns=cols_to_drop)
print(f'\nShape apres suppression : {df.shape}')


EmployeeCount : 1 valeur(s) unique(s) -> [1]
Over18 : 1 valeur(s) unique(s) -> <StringArray>
['Y']
Length: 1, dtype: str
StandardHours : 1 valeur(s) unique(s) -> [8]

Shape apres suppression : (4410, 26)


#### 4. Imputation des valeurs manquantes

- Numeriques -> mediane
- Categorielles -> mode


In [4]:
# Imputation separee (numerique/categorielle) pour limiter la perte d'information.
print('Valeurs manquantes avant imputation :')
missing = df.isnull().sum()
print(missing[missing > 0])
print()

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            print(f'  {col} -> mediane ({med})')
        else:
            mode = df[col].mode()[0]
            df[col] = df[col].fillna(mode)
            print(f'  {col} -> mode ({mode})')

print(f'\nValeurs manquantes apres imputation : {df.isnull().sum().sum()}')


Valeurs manquantes avant imputation :
NumCompaniesWorked         19
TotalWorkingYears           9
EnvironmentSatisfaction    25
JobSatisfaction            20
WorkLifeBalance            38
dtype: int64

  NumCompaniesWorked -> mediane (2.0)
  TotalWorkingYears -> mediane (10.0)
  EnvironmentSatisfaction -> mediane (3.0)
  JobSatisfaction -> mediane (3.0)
  WorkLifeBalance -> mediane (3.0)

Valeurs manquantes apres imputation : 0


#### 5. Encodage Attrition (Yes/No -> 1/0)

La variable cible est encodée en binaire pour faciliter les calculs statistiques et ML.


In [5]:
# Encodage binaire de la cible `Attrition` (Yes/No -> 1/0).
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
print(f'Distribution Attrition :\n{df["Attrition"].value_counts()}')


Distribution Attrition :
Attrition
0    3699
1     711
Name: count, dtype: int64


#### 6. Export et validation

Dernière vérification de cohérence, puis export du dataset propre vers `data/processed/`.


In [6]:
# Validation structurelle puis export de la base nettoyee.
assert df.shape[0] == 4410, f'Nombre de lignes inattendu : {df.shape[0]}'
assert df.isna().sum().sum() == 0, 'Il reste des NaN'
assert 'EmployeeID' in df.columns, 'EmployeeID manquant'

os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path = os.path.join(PROCESSED_DIR, 'attrition_merged_base.csv')
df.to_csv(output_path, index=False)

print(f'Shape   : {df.shape}')
print(f'NaN     : {df.isna().sum().sum()}')
print(f'Colonnes: {df.columns.tolist()}')
print(f'Export  : {output_path}')


Shape   : (4410, 26)
NaN     : 0
Colonnes: ['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeID', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating']
Export  : ..\data\processed\attrition_merged_base.csv


---

#### Mini-conclusion — Notebook 02

**Ce qui a ete fait** :
- Fusion des 3 tables RH sur `EmployeeID` (inner join, 4 410 lignes conservees)
- Suppression de 3 colonnes constantes (`EmployeeCount`, `Over18`, `StandardHours`) : aucune variance, aucun pouvoir discriminant
- Imputation des valeurs manquantes : mediane pour les numeriques, mode pour les categorielles. Choix conservateur limitant la perte d'information.
- Encodage de `Attrition` en binaire (Yes=1, No=0) pour usage en modelisation

**Decisions justifiees** :
- **Imputation mediane** plutot que moyenne : robuste aux outliers (ex: `MonthlyIncome` asymetrique)
- **Mode** pour les categorielles : preserve la distribution originale
- **Pas de suppression de lignes** : le taux de NA est trop faible (< 1 %) pour justifier une perte de donnees

**Prochaine etape** : Notebook 03 — Engineering de la variable `avg_work_hours` a partir de la badgeuse